# Performance Analytics

Generates:
- `fund_scorecard.csv`
- `alpha_beta.csv`
- `benchmark_comparison.png`


In [ ]:
!pip install pandas numpy matplotlib seaborn

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")

# Project root
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Directories
processed_dir = ROOT / "bluestock_mf_capstone" / "data" / "processed"
out_dir = processed_dir

# Load datasets
fund_master = pd.read_csv(processed_dir / "cleaned_01_fund_master.csv")
perf = pd.read_csv(processed_dir / "cleaned_07_scheme_performance.csv")
bench = pd.read_csv(processed_dir / "cleaned_10_benchmark_indices.csv")

# Convert date columns
fund_master["launch_date"] = pd.to_datetime(
    fund_master["launch_date"], errors="coerce"
)

bench["date"] = pd.to_datetime(
    bench["date"], errors="coerce"
)

bench = bench.dropna(subset=["date"])

# Convert numeric columns
numeric_cols = [
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct",
    "aum_crore",
    "expense_ratio_pct",
    "morningstar_rating",
]

for col in numeric_cols:
    if col in perf.columns:
        perf[col] = pd.to_numeric(perf[col], errors="coerce")

# Check dataset shapes
print("Fund Master:", fund_master.shape)
print("Performance:", perf.shape)
print("Benchmark:", bench.shape)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
base = perf.copy()

# Fill reference fields from fund_master if missing in perf
for col in ['scheme_name','fund_house','category','plan','risk_category','expense_ratio_pct']:
    if col not in base.columns and col in fund_master.columns and 'amfi_code' in base.columns:
        base = base.merge(fund_master[['amfi_code', col]], on='amfi_code', how='left')

if 'max_drawdown_pct' in base.columns:
    base['loss_penalty'] = -pd.to_numeric(base['max_drawdown_pct'], errors='coerce')
else:
    base['loss_penalty'] = np.nan

metrics_higher = ['return_3yr_pct','sharpe_ratio','alpha','sortino_ratio','morningstar_rating']
metrics_lower = ['expense_ratio_pct','std_dev_ann_pct','loss_penalty']

def minmax(s):
    s = pd.to_numeric(s, errors='coerce')
    mn = s.min(skipna=True)
    mx = s.max(skipna=True)
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return s * 0
    return (s - mn) / (mx - mn)

for c in metrics_higher:
    base[c + '_mm'] = minmax(base[c]) if c in base.columns else np.nan
for c in metrics_lower:
    base[c + '_mm'] = minmax(base[c]) if c in base.columns else np.nan

higher_parts = [c + '_mm' for c in metrics_higher if c in base.columns]
lower_parts = [c + '_mm' for c in metrics_lower if c in base.columns]

base['score_raw'] = base[higher_parts].sum(axis=1) - base[lower_parts].sum(axis=1)
base['score_mm'] = minmax(base['score_raw'])

score_cols = [
    'amfi_code','scheme_name','fund_house','category','plan','risk_grade','risk_category',
    'return_3yr_pct','sharpe_ratio','alpha','sortino_ratio','expense_ratio_pct','std_dev_ann_pct','max_drawdown_pct',
    'morningstar_rating','score_mm'
]
score_cols = [c for c in score_cols if c in base.columns]
scorecard = base[score_cols].sort_values('score_mm', ascending=False)
scorecard.head(10)

In [ ]:
scorecard.to_csv(out_dir / 'fund_scorecard.csv', index=False)

alpha_beta_cols = [
    'amfi_code','scheme_name','fund_house','category','plan',
    'alpha','beta','sharpe_ratio','return_3yr_pct','max_drawdown_pct','expense_ratio_pct','morningstar_rating','score_mm'
]
alpha_beta_cols = [c for c in alpha_beta_cols if c in base.columns]
alpha_beta = base[alpha_beta_cols].sort_values(['alpha','sharpe_ratio'], ascending=False)
alpha_beta.to_csv(out_dir / 'alpha_beta.csv', index=False)

scorecard.shape, alpha_beta.shape

In [ ]:
preferred = ['NIFTY50','NIFTY 50','NIFTY_50']
bench['index_name'] = bench['index_name'].astype(str)
cands = [b for b in preferred if b in set(bench['index_name'])]
idx_name = cands[0] if cands else bench['index_name'].value_counts().index[0]

bench_idx = bench[bench['index_name'] == idx_name].sort_values('date')
bench_idx = bench_idx.dropna(subset=['close_value','date'])

start_close = float(bench_idx.iloc[0]['close_value'])
end_close = float(bench_idx.iloc[-1]['close_value'])
bench_return_pct = (end_close - start_close) / start_close * 100

fund_median_return = pd.to_numeric(base['return_3yr_pct'], errors='coerce').median(skipna=True)
fund_mean_return = pd.to_numeric(base['return_3yr_pct'], errors='coerce').mean(skipna=True)

plot_df = pd.DataFrame({
    'Series': ['Benchmark (point-to-point)', 'Funds Median (3Y)', 'Funds Mean (3Y)'],
    'Return_%': [bench_return_pct, fund_median_return, fund_mean_return],
})

plt.figure(figsize=(9, 5))
bar = sns.barplot(data=plot_df, x='Series', y='Return_%',
                 palette=['#4c72b0','#55a868','#c44e52'])
bar.set_title(f'Benchmark vs Funds Returns (Index: {idx_name})')
bar.set_xlabel('')
bar.set_ylabel('Return (%)')
plt.xticks(rotation=15, ha='right')
for p in bar.patches:
    bar.annotate(f"{p.get_height():.1f}%", (p.get_x()+p.get_width()/2., p.get_height()),
                 ha='center', va='bottom', fontsize=9,
                 xytext=(0, 3), textcoords='offset points')

plt.tight_layout()
chart_path = out_dir / 'benchmark_comparison.png'
plt.savefig(chart_path, dpi=180)
plt.show()
chart_path